
# Assignment 1: Boolean Model, TF-IDF, and Data Retrieval vs. Information Retrieval Conceptual Questions

**Student names**: _Your_names_here_ <br>
**Group number**: _Your_group_here_ <br>
**Date**: _Submission Date_

## Important notes
Please carefully read the following notes and consider them for the assignment delivery. Submissions that do not fulfill these requirements will not be assessed and should be submitted again.
1. You may work in groups of maximum 2 students.
2. The assignment must be delivered in ipynb format.
3. The assignment must be typed. Handwritten assignments are not accepted.

**Due date**: 18.09.2026 23:59

In this assignment, you will:
- Implement a Boolean retrieval model
- Compute TF-IDF vectors for documents
- Run retrieval on queries
- Answer conceptual questions 

---
## Dataset

You will use the **Cranfield** dataset, provided in this file:

- `cran.all.1400`: The document collection (1400 documents)

**The code to parse the file is ready — just update the cran file path to match your own file location. Use the docs variable in your code for the parsed file**

### Load and parse documents (provided)

Run the cell to parse the Cranfield documents. Update the path so it points to your `cran.all.1400` file.


In [23]:

# Read 'cran.all.1400' and parse the documents into a suitable data structure

CRAN_PATH = r"./task/cran.all.1400"  # <-- change this!

def parse_cranfield(path):
    docs = {}
    current_id = None
    current_field = None
    buffers = {"T": [], "A": [], "B": [], "W": []}
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(".I "):
                if current_id is not None:
                    docs[current_id] = {
                        "id": current_id,
                        "title": " ".join(buffers["T"]).strip(),
                        "abstract": " ".join(buffers["W"]).strip()
                    }
                current_id = int(line.split()[1])
                buffers = {k: [] for k in buffers}
                current_field = None
            elif line.startswith("."):
                tag = line[1:].strip()
                current_field = tag if tag in buffers else None
            else:
                if current_field is not None:
                    buffers[current_field].append(line)
    if current_id is not None:
        docs[current_id] = {
            "id": current_id,
            "title": " ".join(buffers["T"]).strip(),
            "abstract": " ".join(buffers["W"]).strip()
        }
    print(f"Parsed {len(docs)} documents.")
    return docs

docs = parse_cranfield(CRAN_PATH)



Parsed 1400 documents.


## 1.1 – Boolean Retrieval Model

### 1.1.1 Tokenize documents

Implement tokenization using the given list of stopwords. Create a list of normalized terms per document (e.g., lowercase, remove punctuation/digits; drop stopwords). Store the token lists to use in later steps.

In [24]:
# TODO: Implement tokenization using the given list of stopwords, create list of terms per document

STOPWORDS = set("""a about above after again against all am an and any are aren't as at be because been
before being below between both but by can't cannot could couldn't did didn't do does doesn't doing don't down
during each few for from further had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours ourselves out over own same shan't she
she'd she'll she's should shouldn't so some such than that that's the their theirs them themselves then there there's
these they they'd they'll they're they've this those through to too under until up very was wasn't we we'd we'll we're
we've were weren't what what's when when's where where's which while who who's whom why why's with won't would wouldn't
you you'd you'll you're you've your yours yourself yourselves""".split())

# Your code here
import re

def tokenize(text):
    lowercase_text = text.lower()
    text_without_punctuation_and_digits = re.sub(r"[^a-z\s]", " ", lowercase_text)
    raw_tokens = text_without_punctuation_and_digits.split()

    normalized_tokens = []

    for token in raw_tokens:
        if token not in STOPWORDS:
            normalized_tokens.append(token)

    return normalized_tokens

document_tokens = {}

for document_id, document in docs.items():
    full_text = document["title"] + " " + document["abstract"]
    document_tokens[document_id] = tokenize(full_text)

print(f"Tokenized {len(document_tokens)} documents.")


Tokenized 1400 documents.


### Build vocabulary

Create a set (or list) of unique terms from all tokenized documents. Report the number of unique terms.


In [25]:
# TODO: Create a set or list of unique terms

# Report: 
# - Number of unique terms

# Your code here
unique_terms = set()

for document_id, token_sets in document_tokens.items():
    for token in token_sets:
        if token not in unique_terms:
            unique_terms.add(token)

print(f"Number of unique terms: {len(unique_terms)}")


Number of unique terms: 6934


### Build inverted index

For each term, store the list (or set) of document IDs where the term appears.


In [26]:

# TODO: For each term, store list of document IDs where the term appears
# Your code here
inverted_index = {}

for term in unique_terms:
    inverted_index[term] = []

    for document_id, token_sets in document_tokens.items():
        if term in token_sets:
            inverted_index[term].append(document_id)

print(f"Inverted index built with {len(inverted_index)} terms.")


Inverted index built with 6934 terms.


### Retrieve documents for a Boolean query (AND/OR)

Create a function to retrieve documents for a Boolean query (AND/OR) with query terms.  


In [27]:
# TODO: Create a function for retrieving documents for a Boolean query (AND/OR) with query terms

def get_posting_set(term):
    normalized_term = term.strip().lower()

    if normalized_term not in inverted_index:
        return set()
    
    return set(inverted_index[normalized_term])

def boolean_retrieve(query: str):
    matching_document_ids = set()

    or_clauses = query.split("OR")

    for or_clause in or_clauses:
        and_terms = or_clause.split("AND")
        clause_document_ids = None

        for term in and_terms:
            stripped_term = term.strip()

            if stripped_term == "":
                continue

            term_document_ids = get_posting_set(stripped_term)

            if clause_document_ids is None:
                clause_document_ids = term_document_ids
            else:
                clause_document_ids = clause_document_ids.intersection(term_document_ids)

        if clause_document_ids is None:
            continue

        matching_document_ids = matching_document_ids.union(clause_document_ids)

    return sorted(matching_document_ids)


In [28]:
# Do not change this code
boolean_queries = [
  "gas AND pressure",
  "structural AND aeroelastic AND flight AND high AND speed OR aircraft",
  "heat AND conduction AND composite AND slabs",
  "boundary AND layer AND control",
  "compressible AND flow AND nozzle",
  "combustion AND chamber AND injection",
  "laminar AND turbulent AND transition",
  "fatigue AND crack AND growth",
  "wing AND tip AND vortices",
  "propulsion AND efficiency"
]

In [29]:
# Run Boolean queries in batch, using the function you created
def run_batch_boolean(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = boolean_retrieve(q)
        results[f"Q{i}"] = res
    return results

boolean_results = run_batch_boolean(boolean_queries)
for qid, res in boolean_results.items():
    print(qid, "=>", res[:5])


Q1 => [27, 49, 85, 101, 110]
Q2 => [12, 14, 29, 47, 51]
Q3 => [5, 399]
Q4 => [1, 61, 244, 265, 342]
Q5 => [118, 131]
Q6 => []
Q7 => [7, 9, 80, 89, 96]
Q8 => []
Q9 => [675]
Q10 => [968]


## Part 1.2 – TF-IDF Indexing


$tf_{i,j} = \text{Raw Frequency}$

$idf_t = \log\left(\frac{N}{df_t}\right)$

### Build document–term matrix (TF and IDF weights)

Compute tf and idf using the formulas above and store the weights in a document–term matrix (rows = documents, columns = terms).



In [30]:
# TODO: Calculate the weights for the documents and the terms using tf and idf weighting. Put these values into a document–term matrix (rows = documents, columns = terms).

# Your code here
import math
import numpy as np

def tf(term, document_id):
    return document_tokens[document_id].count(term)

def idf(term):
    return math.log(len(docs) / len(inverted_index[term]))

vocabulary = sorted(unique_terms)
term_to_column = {}

for column_index, term in enumerate(vocabulary):
    term_to_column[term] = column_index

document_ids = sorted(docs.keys())
document_id_to_row = {}

for row_index, document_id in enumerate(document_ids):
    document_id_to_row[document_id] = row_index

number_of_documents = len(document_ids)
number_of_terms = len(vocabulary)

idf_weights = np.zeros(number_of_terms)

for term, column_index in term_to_column.items():
    idf_weights[column_index] = idf(term)

tf_matrix = np.zeros((number_of_documents, number_of_terms))

for document_id, tokens in document_tokens.items():
    row_index = document_id_to_row[document_id]
    term_counts = {}

    for token in tokens:
        if token not in term_counts:
            term_counts[token] = 0

        term_counts[token] += 1

    for term, raw_frequency in term_counts.items():
        column_index = term_to_column[term]
        tf_matrix[row_index, column_index] = raw_frequency

document_term_matrix = tf_matrix * idf_weights

print("Document-term matrix shape:", document_term_matrix.shape)




Document-term matrix shape: (1400, 6934)


### Build TF–IDF document vectors

From the matrix, build a TF–IDF vector for each document (consider normalization if needed for cosine similarity).


In [31]:

# TODO: Build TF–IDF document vectors from the document–term matrix
# Your code here

tfidf_document_vectors = {}

for document_id in document_ids:
    row_index = document_id_to_row[document_id]
    vector = document_term_matrix[row_index]
    vector_norm = np.linalg.norm(vector)

    if vector_norm == 0:
        tfidf_document_vectors[document_id] = vector
    else:
        tfidf_document_vectors[document_id] = vector / vector_norm

print("Built", len(tfidf_document_vectors), "L2-normalized TF-IDF document vectors.")
print("Document 1 vector length:", np.linalg.norm(tfidf_document_vectors[1]))



Built 1400 L2-normalized TF-IDF document vectors.
Document 1 vector length: 1.0


### Implement cosine similarity

Implement a function to compute cosine similarity scores between a (tokenized) query and all documents.


In [32]:

# TODO: Create a function for calculating the similarity score of all the documents by their relevance to query terms

def build_query_vector(query):
    query_tokens = tokenize(query)
    query_vector = np.zeros(number_of_terms)

    term_counts = {}
    for token in query_tokens:
        if token not in term_to_column:
            continue

        if token not in term_counts:
            term_counts[token] = 0
        term_counts[token] += 1

    for term, raw_frequency in term_counts.items():
        column_index = term_to_column[term]
        query_vector[column_index] = raw_frequency * idf_weights[column_index]

    vector_norm = np.linalg.norm(query_vector)
    if vector_norm == 0:
        return query_vector

    return query_vector / vector_norm

def tfidf_retrieve(query: str):
    # Your code here
    query_vector = build_query_vector(query)
    scored_documents = []

    for document_id, document_vector in tfidf_document_vectors.items():
        similarity_score = float(np.dot(query_vector, document_vector))
        scored_documents.append((document_id, similarity_score))

    scored_documents.sort(key=lambda document_score: document_score[1], reverse=True)

    ranked_document_ids = []
    for document_id, similarity_score in scored_documents:
        ranked_document_ids.append(document_id)

    return ranked_document_ids


In [33]:
# Do not change this code
tfidf_queries = [
  "gas pressure",
  "structural aeroelastic flight high speed aircraft",
  "heat conduction composite slabs",
  "boundary layer control",
  "compressible flow nozzle",
  "combustion chamber injection",
  "laminar turbulent transition",
  "fatigue crack growth",
  "wing tip vortices",
  "propulsion efficiency"
]

In [34]:
# Run TF-IDF queries in batch (print top-5 results for each), using the function you created
def run_batch_tfidf(queries):
    results = {}
    for i, q in enumerate(queries, 1):
        res = tfidf_retrieve(q)
        results[f"Q{i}"] = res
    return results

tfidf_results = run_batch_tfidf(tfidf_queries)

for qid, res in tfidf_results.items():
    print(qid, "=>", res[:5])


Q1 => [169, 1286, 167, 185, 1003]
Q2 => [12, 51, 746, 875, 884]
Q3 => [399, 144, 485, 5, 181]
Q4 => [368, 748, 638, 451, 1349]
Q5 => [389, 118, 1187, 172, 173]
Q6 => [974, 628, 397, 308, 635]
Q7 => [418, 1264, 315, 272, 9]
Q8 => [768, 726, 1196, 883, 884]
Q9 => [1284, 433, 675, 1271, 288]
Q10 => [968, 1328, 1380, 1092, 592]



## Part 1.3 – Conceptual Questions

Answer the following questions:

**1. What is the difference between data retrieval and information retrieval?**
Data retrieval looks up exact matches in structured data, for example a database query like attribute = value. The result is either correct or not. If the query does not match exactly, you do not get a useful answer. 

Information retrieval is about satisfying an information need over a collection of documents. The query is often vague or in natural language, and the system ranks documents by how relevant they seem, instead of only returning exact matches.

**For the following scenarios, which approach would be suitable data retrieval or information retrieval? Explain your reasoning.** <br>
1.a A clerk in pharmacy uses the following query: Medicine_name = Ibuprofen_400mg
Data retrieval would be suitable, because the query is for a specific medicine name in a structured field. This would be easy to look up in a database, and would provide a specific answer.

1.b A clerk in pharmacy uses the following query: An anti-biotic medicine
Information retrieval would be suitable. This is not an exact lookup on one field. There could be many different antibiotics, and there is no single correct row, so the system would need to rank possible medicines by relevance.

1.c Searching for the schedule of a flight using the following query: Flight_ID = ZEFV2
Data retrieval would be suitable, because the query is for a specific flight ID. This is trivial to look up in a database, and would provide a specific answer.

1.d Searching an E-commerce website using the following query to find an specific shoe: Brooks Ghost 15
Information retrieval would be suitable. Even though it is a specific shoe, this is still a search query, not a structured lookup like Product_name = Brooks Ghost 15. The site would match terms and rank results, for example different colors, sizes, or sellers.

1.e Searching the same E-commerce website using the following query: Nice running shoes
Information retrieval would be suitable. This is a vague query, and "nice" is subjective, so there is no exact record to look up. The system would have to rank shoes by how relevant they seem.
